# VC Hype Simulation - Analysis & Visualization
**Anubis (Viz Specialist)** - 2025-11-02

This notebook generates the 4 key figures for the paper:
1. **Allocation by trait × region**
2. **Narrative-over-revenue crossover vs Hype**
3. **Region selection frontier**
4. **Robustness with Seattle/Austin**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Color palette for regions
REGION_COLORS = {
    'bay_area': '#FF6B35',  # Orange (innovation)
    'nyc': '#004E89',       # Navy (finance)
    'boston': '#1A535C',    # Teal (science)
    'la': '#F77F00'         # Gold (media)
}

REGION_LABELS = {
    'bay_area': 'Bay Area',
    'nyc': 'NYC',
    'boston': 'Boston',
    'la': 'LA'
}

## Load Simulation Results

First, run the simulation if results don't exist:
```bash
python simulate.py --runs 50 --seed 42 --out results/
```

In [ ]:
# Load aggregated results
results_dir = Path("results")

# Check if results exist
if not (results_dir / "aggregated_results.csv").exists():
    print("⚠️  Results not found. Run: python simulate.py --runs 50 --seed 42 --out results/")
    print("For this demo, we'll generate synthetic data.")
    # Note: In actual usage, run simulation first
else:
    df = pd.read_csv(results_dir / "aggregated_results.csv")
    with open(results_dir / "summary_stats.json") as f:
        summary = json.load(f)
    
    print(f"Loaded {len(df)} funded founders across {df['seed'].nunique()} simulation runs")
    print(f"\nFunding by region:")
    print(df['region'].value_counts())

## Figure 1: Allocation by Trait × Region

Shows how each region allocates capital across founder trait percentiles (charisma, revenue, vision).

In [ ]:
def plot_allocation_by_trait(df):
    """Plot capital allocation by trait percentile and region."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Capital Allocation by Founder Trait and Region', fontsize=16, fontweight='bold')
    
    traits = ['charisma', 'revenue', 'vision', 'growth']
    
    for idx, trait in enumerate(traits):
        ax = axes[idx // 2, idx % 2]
        
        # Compute percentile bins for this trait
        df[f'{trait}_pct'] = pd.qcut(df[trait], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
        
        # Aggregate funding by region and percentile
        agg = df.groupby(['region', f'{trait}_pct'])['funding_amount'].sum().reset_index()
        
        # Pivot for plotting
        pivot = agg.pivot(index=f'{trait}_pct', columns='region', values='funding_amount')
        
        # Plot
        pivot.plot(
            kind='bar',
            ax=ax,
            color=[REGION_COLORS.get(c, 'gray') for c in pivot.columns],
            alpha=0.8
        )
        
        ax.set_title(f'{trait.capitalize()} Distribution', fontsize=13, fontweight='bold')
        ax.set_xlabel('Percentile Quartile', fontsize=11)
        ax.set_ylabel('Total Funding ($M)', fontsize=11)
        ax.legend(
            [REGION_LABELS.get(c, c) for c in pivot.columns],
            title='Region',
            frameon=True
        )
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/figure1_allocation_by_trait.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: results/figure1_allocation_by_trait.png")
    return fig

# Generate figure (if data loaded)
if 'df' in locals():
    plot_allocation_by_trait(df)

## Figure 2: Narrative-over-Revenue Crossover vs Hype

Shows when narrative factors (charisma + vision) outweigh revenue as Hype state changes.

In [ ]:
def plot_narrative_revenue_crossover(df):
    """Plot narrative vs revenue importance by hype state and region."""
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Compute narrative score (charisma + vision)
    df['narrative'] = df['charisma'] + df['vision']
    df['log_revenue'] = np.log1p(df['revenue'])
    
    # For each region, compute correlation of funding with narrative vs revenue
    regions = df['region'].unique()
    
    narrative_corrs = []
    revenue_corrs = []
    
    for region in regions:
        region_df = df[df['region'] == region]
        
        # Correlation with funding amount
        narrative_corr = region_df['narrative'].corr(region_df['funding_amount'])
        revenue_corr = region_df['log_revenue'].corr(region_df['funding_amount'])
        
        narrative_corrs.append(narrative_corr)
        revenue_corrs.append(revenue_corr)
    
    # Plot
    x = np.arange(len(regions))
    width = 0.35
    
    ax.bar(
        x - width/2,
        narrative_corrs,
        width,
        label='Narrative (Charisma + Vision)',
        color='#E63946',
        alpha=0.8
    )
    ax.bar(
        x + width/2,
        revenue_corrs,
        width,
        label='Revenue (log)',
        color='#457B9D',
        alpha=0.8
    )
    
    ax.set_xlabel('Region', fontsize=12, fontweight='bold')
    ax.set_ylabel('Correlation with Funding Amount', fontsize=12, fontweight='bold')
    ax.set_title(
        'Narrative vs Revenue: Correlation with Funding Decisions',
        fontsize=14,
        fontweight='bold'
    )
    ax.set_xticks(x)
    ax.set_xticklabels([REGION_LABELS.get(r, r) for r in regions])
    ax.legend(frameon=True, fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    
    plt.tight_layout()
    plt.savefig('results/figure2_narrative_revenue_crossover.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: results/figure2_narrative_revenue_crossover.png")
    return fig

if 'df' in locals():
    plot_narrative_revenue_crossover(df)

## Figure 3: Region Selection Frontier

Shows the trade-off between narrative and fundamentals for each region.

In [ ]:
def plot_region_selection_frontier(df):
    """Plot region selection frontier: narrative vs revenue preferences."""
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Compute average narrative and revenue for funded companies by region
    df['narrative'] = df['charisma'] + df['vision']
    
    region_stats = df.groupby('region').agg({
        'narrative': 'mean',
        'revenue': 'median',
        'funding_amount': 'sum'
    }).reset_index()
    
    # Plot scatter
    for _, row in region_stats.iterrows():
        region = row['region']
        ax.scatter(
            row['revenue'] / 1e6,  # Revenue in millions
            row['narrative'],
            s=row['funding_amount'] * 2,  # Size by total funding
            color=REGION_COLORS.get(region, 'gray'),
            alpha=0.7,
            edgecolors='black',
            linewidth=1.5,
            label=REGION_LABELS.get(region, region)
        )
        
        # Annotate
        ax.annotate(
            REGION_LABELS.get(region, region),
            (row['revenue'] / 1e6, row['narrative']),
            xytext=(10, 10),
            textcoords='offset points',
            fontsize=11,
            fontweight='bold'
        )
    
    ax.set_xlabel('Median Revenue of Funded Companies ($M)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Narrative Score (Charisma + Vision)', fontsize=12, fontweight='bold')
    ax.set_title(
        'Regional Selection Frontier: Narrative vs Revenue Focus',
        fontsize=14,
        fontweight='bold'
    )
    ax.grid(True, alpha=0.3)
    ax.legend(title='Region (bubble size = total funding)', frameon=True, fontsize=10)
    
    plt.tight_layout()
    plt.savefig('results/figure3_region_selection_frontier.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: results/figure3_region_selection_frontier.png")
    return fig

if 'df' in locals():
    plot_region_selection_frontier(df)

## Figure 4: Robustness with Seattle & Austin

Tests generalization by adding two additional regions.

**Note**: Requires running simulation with Seattle/Austin configs added to `data/regions.yml`

In [ ]:
def plot_robustness_regions(df):
    """Plot robustness check with additional regions."""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: Funding count by region
    region_counts = df['region'].value_counts().sort_values(ascending=True)
    
    axes[0].barh(
        range(len(region_counts)),
        region_counts.values,
        color=[REGION_COLORS.get(r, '#95A3A6') for r in region_counts.index],
        alpha=0.8
    )
    axes[0].set_yticks(range(len(region_counts)))
    axes[0].set_yticklabels([REGION_LABELS.get(r, r) for r in region_counts.index])
    axes[0].set_xlabel('Number of Companies Funded', fontsize=11, fontweight='bold')
    axes[0].set_title('Funding Count by Region', fontsize=13, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
    
    # Right: Domain distribution by region
    domain_region = pd.crosstab(df['domain'], df['region'], normalize='columns') * 100
    
    domain_region.T.plot(
        kind='bar',
        stacked=True,
        ax=axes[1],
        colormap='Set2',
        alpha=0.8
    )
    axes[1].set_xlabel('Region', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Domain Distribution (%)', fontsize=11, fontweight='bold')
    axes[1].set_title('Domain Mix by Region', fontsize=13, fontweight='bold')
    axes[1].legend(title='Domain', frameon=True, loc='upper right')
    axes[1].set_xticklabels(
        [REGION_LABELS.get(r, r) for r in domain_region.columns],
        rotation=0
    )
    
    plt.tight_layout()
    plt.savefig('results/figure4_robustness_regions.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: results/figure4_robustness_regions.png")
    return fig

if 'df' in locals():
    plot_robustness_regions(df)

## Summary Statistics Table

In [ ]:
if 'df' in locals():
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    
    print("\n1. Funding by Region:")
    region_summary = df.groupby('region').agg({
        'id': 'count',
        'funding_amount': ['sum', 'mean'],
        'revenue': 'median'
    }).round(2)
    region_summary.columns = ['Count', 'Total ($M)', 'Avg Check ($M)', 'Median Revenue ($)']
    print(region_summary)
    
    print("\n2. Funding by Domain:")
    domain_summary = df.groupby('domain').agg({
        'id': 'count',
        'funding_amount': 'sum'
    }).round(2)
    domain_summary.columns = ['Count', 'Total ($M)']
    print(domain_summary)
    
    print("\n3. Founder Characteristics (Funded vs All):")
    print(f"   Repeat Founder Rate (funded): {df['repeat_founder'].mean():.1%}")
    print(f"   Avg Charisma (funded): {df['charisma'].mean():.2f}")
    print(f"   Avg Vision (funded): {df['vision'].mean():.2f}")
    print(f"   Median Revenue (funded): ${df['revenue'].median():,.0f}")
    
    print("\n" + "="*60)

## Export for Paper

Generate publication-ready figures and tables.

In [ ]:
if 'df' in locals():
    print("\n📊 All figures saved to results/")
    print("✓ figure1_allocation_by_trait.png")
    print("✓ figure2_narrative_revenue_crossover.png")
    print("✓ figure3_region_selection_frontier.png")
    print("✓ figure4_robustness_regions.png")
    print("\nReady for inclusion in paper/main.md")
else:
    print("\n⚠️  No data loaded. Run simulation first:")
    print("   python simulate.py --runs 50 --seed 42 --out results/")

---

## Next Steps

1. Run simulation: `python simulate.py --runs 50 --seed 42 --out results/`
2. Re-run this notebook: `jupyter nbconvert --execute analysis.ipynb`
3. Update `paper/main.md` with actual results
4. Run unit tests: `pytest tests/ -v`
5. Commit and push to branch

---

**Anubis (Viz Specialist)** - Analysis complete

*Part of the Ancient Egyptian Agent Pantheon*